# Hill Cipher Encryption
## 1. Let's first set the alphabet in numerical order using a dictionary and a reversed version for number to alphabet.


In [107]:
alphabet = {chr(ord('A') + i): i for i in range(26)}
reversed_alphabet_dict = {v: k for k, v in alphabet.items()}  
for key in alphabet:
    print(key, alphabet[key])

A 0
B 1
C 2
D 3
E 4
F 5
G 6
H 7
I 8
J 9
K 10
L 11
M 12
N 13
O 14
P 15
Q 16
R 17
S 18
T 19
U 20
V 21
W 22
X 23
Y 24
Z 25


## 2. Find determinant and GCD for key validation 
Next is a key. Using the formula Cipher = Key * Plaintext (mod 26), we can encrypt our plain text with a n x n matrix, where n >= 2 and it has the gcd(det, 26) = 1.

So we need:
find_det(key)

Using gaussian elimination method.

In [108]:
def find_det(matrix):
    # Get the size of the matrix (n x n)
    n = len(matrix)
    
    # Make a copy of the matrix so we do not alter the original data
    mat = [row[:] for row in matrix]
    
    det_sign = 1  # Tracks sign changes due to row swaps

    for i in range(n):  
        # Find a pivot in the current column
        pivot_row = i
        
        # If all numbers in this column are zero, the determinant is 0
        while pivot_row < n and mat[pivot_row][i] == 0:
            pivot_row += 1
            
        if pivot_row == n:
            return 0

        # If the pivot is not in the current row, swap rows
        if pivot_row != i:
            mat[i], mat[pivot_row] = mat[pivot_row], mat[i]
            det_sign *= -1  # Each row swap flips the sign of the determinant

        # Eliminate all values below the pivot
        for j in range(i + 1, n):
            # Calculate the multiplier to make the entry zero
            factor = mat[j][i] / mat[i][i]
            for k in range(i, n):
                mat[j][k] -= factor * mat[i][k]

    # Multiply the numbers along the main diagonal
    determinant = det_sign
    for i in range(n):
        determinant *= mat[i][i]

    return determinant

# Example usage with a 2x2 matrix
key = [
    [3, 3],
    [2, 5]
]

det = find_det(key)
print("Determinant:", det)

Determinant: 9.0


In [109]:

def find_gcd(det, num1=26):
    num = num1
    denom = det
    r = num % denom
    
    while r != 0:
        num, denom = denom, r
        r = num % denom
        
    return denom;
            
        
print("GCD:", find_gcd(det))
        

GCD: 1.0


## 3. Divide the plaintext into n x len(plaintext) matrix and turn into column vector.

In [110]:
def block_plaintext(plaintext, n):
    if n <= 0:
        raise ValueError("n must be positive")

    cleaned = ''.join(
        char for char in plaintext.upper()
        if char in alphabet
    )

    # Pad the final block with X
    padding = (-len(cleaned)) % n
    cleaned += "X" * padding

    return [
        [alphabet[char] for char in cleaned[i:i + n]]
        for i in range(0, len(cleaned), n)
    ]



plaintext = "hello"
blocks = block_plaintext(plaintext, 2)
print(blocks)

def vector_block(block):
    return [[[val] for val in b] for b in block]

vector_blocks = vector_block(blocks)
print(vector_blocks)

[[7, 4], [11, 11], [14, 23]]
[[[7], [4]], [[11], [11]], [[14], [23]]]


## 4. Multiply key matrix and blocked translated plaintext

We multiply using our key matrix as a multiple to our blocked plaintext then finding the mod 26 of the product. Revert back to its alphabetical equivalent. 

In [111]:
def multiply_column_block(col_vector, key_matrix):
    """
    Multiplies a square key matrix by a column vector block (K x P) 
    and applies modulo 26.
    """
    n = len(key_matrix)
    # Initialize an empty column vector for the result
    result = [[0] for _ in range(n)]
    
    for i in range(n):
        row_sum = 0
        for k in range(n):
            # Key row i, column k multiplied by column vector row k, column 0
            row_sum += key_matrix[i][k] * col_vector[k][0]
            
        result[i][0] = row_sum % 26
        
    return result


def multiply_blocks(col_vect_blocks, key_matrix):
    return [multiply_column_block(b, key_matrix) for b in col_vect_blocks]

encrypted_blocks = multiply_blocks(vector_blocks, key)


def number_to_alphabet(blocks):
    return "".join(
    reversed_alphabet_dict[value[0]]
    for block in blocks
    for value in block
)
        
cipher_text = number_to_alphabet(encrypted_blocks)

print("Cipher text: ", cipher_text)



Cipher text:  HIOZHN


## 5. Place it all in one function with validation

In [112]:
def hill_cipher_encryption(plaintext, key_matrix):
    if find_gcd(find_det(key_matrix), 26) != 1:
        print("Key matrix is not valid. Its determinant is not a co-prime of 26")
        raise ValueError
    
    for row in range(len(key_matrix) - 1):
        if len(key_matrix[row]) != len(key_matrix[row + 1]):
            print("Key matrix must be a square matrix")
            raise ValueError
        
        
    n = len(key_matrix)
    
    plaintext_block = block_plaintext(plaintext, n)
    col_vect = vector_block(plaintext_block)
    encrypted_blocks = multiply_blocks(col_vect, key_matrix)
    cipher_text = number_to_alphabet(encrypted_blocks)
    
    return cipher_text
    

key = [
    [3, 3],
    [2, 5]
]

plaintext = "I have the intel find me"
cipher  = hill_cipher_encryption(plaintext, key)
print("Cipher: ", cipher)

Cipher:  TZLBRZHILDRGWVLDTODT


# Hill Cipher Decryption
Procedure:
1. Turn your cipher text to numerics
2. Group into column vectors
3. Find the inverse of the key matrix
4. Multiply using the inverse key matrix
5. Translate back to plain text

## 1. Turn cipher to numbers and block into n

In [113]:
block_cipher = block_plaintext(cipher, n=2)
print(block_cipher)


[[19, 25], [11, 1], [17, 25], [7, 8], [11, 3], [17, 6], [22, 21], [11, 3], [19, 14], [3, 19]]


## 2. Group into Column Vectors

In [114]:
col_vect = vector_block(block_cipher)
print("Column vector: ", col_vect)

Column vector:  [[[19], [25]], [[11], [1]], [[17], [25]], [[7], [8]], [[11], [3]], [[17], [6]], [[22], [21]], [[11], [3]], [[19], [14]], [[3], [19]]]


## 3. Find the inverse of key matrix
- find the mod inverse of the determinant 
- invert the 2x2 matrix

In [115]:
def mod_inverse(det, mod=26):
    """Finds the modular inverse of det modulo 26."""
    det = det % mod
    for x in range(1, mod):
        if (det * x) % mod == 1:
            return x
    raise ValueError(f"Determinant {det} has no modular inverse modulo {mod}.")

def invert_2x2_matrix(key_matrix, mod=26):
    """
    Computes the modular inverse of a 2x2 key matrix modulo 26.
    """
    a = key_matrix[0][0]
    b = key_matrix[0][1]
    c = key_matrix[1][0]
    d = key_matrix[1][1]
    
    # 1. Calculate determinant
    det = (a * d - b * c) % mod
    
    # 2. Find modular inverse of the determinant
    det_inv = mod_inverse(det, mod)
    
    # 3. Form the adjugate matrix and multiply by det_inv modulo 26
    # Adjugate structure: [[d, -b], [-c, a]]
    inv_matrix = [
        [(d * det_inv) % mod, (-b * det_inv) % mod],
        [(-c * det_inv) % mod, (a * det_inv) % mod]
    ]
    
    return inv_matrix


key = [
    [3, 3],
    [2, 5]
]

inverted_key = invert_2x2_matrix(key)
print("Inverted matrix key: ", inverted_key)

Inverted matrix key:  [[15, 17], [20, 9]]


## 4. Multiply Each Ciphertext Block
- Multiply the inverse key matrix by each ciphertext column vector block.
- Apply modulo 26 to each resulting vector element to keep the numbers within the 0 to 25 alphabet range.

In [116]:
decrypted_blocks = multiply_blocks(col_vect, inverted_key)
print("decrypted blocks: ", decrypted_blocks)

decrypted blocks:  [[[8], [7]], [[0], [21]], [[4], [19]], [[7], [4]], [[8], [13]], [[19], [4]], [[11], [5]], [[8], [13]], [[3], [12]], [[4], [23]]]


## 5. Back to plain text



In [117]:
decrypted_text = number_to_alphabet(decrypted_blocks)
print(decrypted_text)

IHAVETHEINTELFINDMEX


In [118]:
def cipher_hill_decryption(cipher, key_matrix):
    if find_gcd(find_det(key_matrix), 26) != 1:
        print("Key matrix is not valid. Its determinant is not coprime to 26")
        raise ValueError

    for row in range(len(key_matrix) - 1):
        if len(key_matrix[row]) != len(key_matrix[row + 1]):
            print("Key matrix must be a square matrix")
            raise ValueError

    n = len(key_matrix)

    # Continue the rest of your decryption code here
        
    plaintext_block = block_plaintext(cipher, n)
    col_vect = vector_block(plaintext_block)
    inverted_key = invert_2x2_matrix(key_matrix)
    decrypted_blocks = multiply_blocks(col_vect, inverted_key)
    decrypted_text = number_to_alphabet(decrypted_blocks)
        
    return decrypted_text

cipher = "TZLBRZHILDRGWVLDTODT"
key = [
    [3, 3],
    [2, 5]
]

decrypted_text = cipher_hill_decryption(cipher, key)
print(decrypted_text)

IHAVETHEINTELFINDMEX
